# 9 WorkFlow Analista Jr, futuro=JULIO  —  Experimento 5B3 (Valores prepandemia)

**Experimento 5B3 - Valores prepandemia**
<br>Cuando un lag apunta a un mes de pandemia (202003 a 202007) su valor se reemplaza por el valor del mismo cliente en un mes prepandemia fijo: lag1 -> 202002, lag3 -> 201912, lag6 -> 201909.

* Training [201901, 202103], validation 202105, final train hasta 202105, produccion 202107
* Grid Search de 30 combinaciones para cada una de las 10 semillas
* Segunda corrida: todos los archivos de salida terminan en `_5B3_Second_Run` y se graban en la carpeta `WF5B3_Second_Run`

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Nov 03 01:25:35 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671183,35.9,1479647,79.1,1479647,79.1
Vcells,1242599,9.5,8388608,64.0,1978693,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

#### Parametros

In [4]:
PARAM <- list()
PARAM$semilla_primigenia <- 701669

# las 10 semillas del experimento (las mismas para 5B1, 5B2, 5B3 y 5B4)
PARAM$semillas <- c(
  701669, 743027, 766957, 780523, 795493,
  163741, 195401, 751643, 959473, 983327
)

PARAM$experimento <- "5B3"
PARAM$experimento_descripcion <- "5B3 - Valores prepandemia"

# identificador de la corrida: carpeta y archivos nuevos, no pisa los resultados de la primera corrida
PARAM$corrida <- "Second_Run"

# sufijo de TODOS los archivos de salida, para que un experimento (o una corrida) no pise a otro
#  todos los archivos terminan en  _<experimento>_Second_Run.<ext>
PARAM$sufijo <- paste0("_", PARAM$experimento, "_", PARAM$corrida)

PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

stopifnot( length(PARAM$semillas) == 10,
           length(unique(PARAM$semillas)) == 10,
           PARAM$semillas[1] == PARAM$semilla_primigenia )

#### Carpeta del Experimento

In [5]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento, "_", PARAM$corrida)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [6]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [7]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




In [8]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [9]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [10]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [11]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [12]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [13]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [14]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [15]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [16]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [17]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [18]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [19]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [20]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [21]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [22]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [23]:
# funcion provista por alumnos amantes de la Estadistica Clasica

drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(get(campo), na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}

In [24]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                      "mrentabilidad_annual"              
 [3] "mcomisiones"                        "mactivos_margen"                   
 [5] "mpasivos_margen"                    "mcuenta_corriente"                 
 [7] "mcaja_ahorro"                       "mcuentas_saldo"                    
 [9] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
[11] "mprestamos_personales"              "mpayroll"                          
[13] "mttarjeta_visa_debitos_automaticos" "mcomisiones_mantenimiento"         
[15] "mtransferencias_recibidas"          "Master_mfinanciacion_limite"       
[17] "Master_msaldototal"                 "Master_mlimitecompra"              
[19] "Master_mconsumototal"               "Master_mpagominimo"                
[21] "Visa_mfinanciacion_limite"          "Visa_msaldototal"                  
[23] "Visa_mlimitecompra"                 "Visa_mconsumototal"                
[25] "Visa_mpagominimo"

In [25]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


inicio drift_deflacion()
fin drift_deflacion()


In [26]:
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"

In [27]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [28]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [29]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"                     "kmes"                              
[57] "mpayroll_sobre_edad"

#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [30]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag3  lags de orden 3
* delta3  = valor actual - lag3
* lag6  lags de orden 6
* delta6  = valor actual - lag6

Los ordenes de lag 1, 3 y 6 son los que define el diseño experimental del Grupo B (5B1 a 5B4).
<br>Aqui se construyen los lags y deltas **sin ningun tratamiento de pandemia** (es el caso base 5B1).
<br>El tratamiento especifico de cada experimento se aplica en la seccion 9.3.2.2 Armado de features.

(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [31]:
# Feature Engineering Historico

# ordenes de lag del diseño experimental del Grupo B
PARAM$FEhist$lags <- c(1, 3, 6)

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html
# el dataset esta ordenado por <numero_de_cliente, foto_mes> desde el Data Drifting

# lags de orden 1, 3 y 6
for (k in PARAM$FEhist$lags)
{
  dataset[,
      paste0(cols_lagueables, "_lag", k) := shift(.SD, k, NA, "lag"),
      by = numero_de_cliente,
      .SDcols = cols_lagueables
  ]
}

# agrego los delta lags
for (vcol in cols_lagueables)
{
  for (k in PARAM$FEhist$lags)
    dataset[, paste0(vcol, "_delta", k) := get(vcol) - get(paste0(vcol, "_lag", k))]
}

Verificacion de los campos recien creados

In [32]:
ncol(dataset)
colnames(dataset)

[1] 381

[1] "numero_de_cliente"                        
  [2] "foto_mes"                                 
  [3] "internet"                                 
  [4] "cliente_edad"                             
  [5] "cliente_antiguedad"                       
  [6] "mrentabilidad"                            
  [7] "mrentabilidad_annual"                     
  [8] "mcomisiones"                              
  [9] "mactivos_margen"                          
 [10] "mpasivos_margen"                          
 [11] "cproductos"                               
 [12] "mcuenta_corriente"                        
 [13] "mcaja_ahorro"                             
 [14] "cdescubierto_preacordado"                 
 [15] "mcuentas_saldo"                           
 [16] "ctarjeta_visa"                            
 [17] "ctarjeta_visa_transacciones"              
 [18] "mtarjeta_visa_consumo"                    
 [19] "ctarjeta_master"                          
 [20] "ctarjeta_master_transacciones"            
 [21] "mtarjeta_master_consumo"                  
 [22] "cprestamos_personales"                    
 [23] "mprestamos_personales"                    
 [24] "cpayroll_trx"                             
 [25] "mpayroll"                                 
 [26] "mttarjeta_visa_debitos_automaticos"       
 [27] "ccomisiones_mantenimiento"                
 [28] "mcomisiones_mantenimiento"                
 [29] "ccomisiones_otras"                        
 [30] "mtransferencias_recibidas"                
 [31] "ccallcenter_transacciones"                
 [32] "thomebanking"                             
 [33] "chomebanking_transacciones"               
 [34] "ctrx_quarter"                             
 [35] "Master_status"                            
 [36] "Master_mfinanciacion_limite"              
 [37] "Master_Fvencimiento"                      
 [38] "Master_msaldototal"                       
 [39] "Master_mlimitecompra"                     
 [40] "Master_fultimo_cierre"                    
 [41] "Master_fechaalta"                         
 [42] "Master_mconsumototal"                     
 [43] "Master_cconsumos"                         
 [44] "Master_mpagominimo"                       
 [45] "Visa_status"                              
 [46] "Visa_mfinanciacion_limite"                
 [47] "Visa_Fvencimiento"                        
 [48] "Visa_msaldototal"                         
 [49] "Visa_mlimitecompra"                       
 [50] "Visa_fultimo_cierre"                      
 [51] "Visa_fechaalta"                           
 [52] "Visa_mconsumototal"                       
 [53] "Visa_cconsumos"                           
 [54] "Visa_mpagominimo"                         
 [55] "clase_ternaria"                           
 [56] "kmes"                                     
 [57] "mpayroll_sobre_edad"                      
 [58] "internet_lag1"                            
 [59] "cliente_edad_lag1"                        
 [60] "cliente_antiguedad_lag1"                  
 [61] "mrentabilidad_lag1"                       
 [62] "mrentabilidad_annual_lag1"                
 [63] "mcomisiones_lag1"                         
 [64] "mactivos_margen_lag1"                     
 [65] "mpasivos_margen_lag1"                     
 [66] "cproductos_lag1"                          
 [67] "mcuenta_corriente_lag1"                   
 [68] "mcaja_ahorro_lag1"                        
 [69] "cdescubierto_preacordado_lag1"            
 [70] "mcuentas_saldo_lag1"                      
 [71] "ctarjeta_visa_lag1"                       
 [72] "ctarjeta_visa_transacciones_lag1"         
 [73] "mtarjeta_visa_consumo_lag1"               
 [74] "ctarjeta_master_lag1"                     
 [75] "ctarjeta_master_transacciones_lag1"       
 [76] "mtarjeta_master_consumo_lag1"             
 [77] "cprestamos_personales_lag1"               
 [78] "mprestamos_personales_lag1"               
 [79] "cpayroll_trx_lag1"                        
 [80] "mpayroll_lag1"                            
 [

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [33]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202107  obviamente completo  (evaluacion de produccion)

* final_train =  [ 201901, 202105 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202105   completo, sin undersampling
   * training = [ 201901, 202103 ]  donde se consideran el 100% de los CONTINUA

En esta seccion solo se define la particion (que registro va a cada conjunto).
<br>Los `lgb.Dataset` de training y validation se construyen al comienzo de 9.3.2.3, una vez que en 9.3.2.2 quedaron armadas las features del experimento.

In [34]:
PARAM$trainingstrategy$validate <- c(202105)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103
)


PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [35]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [36]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]

dataset[, list(registros_train = sum(fold_train)), by = foto_mes][order(foto_mes)]

foto_mes,registros_train
<int>,<int>
201901,24956
201902,25165
201903,25254
201904,25346
201905,25460
201906,25755
201907,26040
201908,26443
201909,26767


#### 9.3.2.2 Armado de features

Experimento **5B3 - Valores prepandemia**
<br>Cuando un lag apunta a un mes de pandemia (202003 a 202007) su valor se reemplaza por el valor del mismo cliente en un mes prepandemia fijo: lag1 -> 202002, lag3 -> 201912, lag6 -> 201909.

Reglas comunes a los cuatro experimentos:
* meses de pandemia = 202003, 202004, 202005, 202006, 202007
* el mes al que apunta cada lag se determina con el mismo `shift()` usado en 9.3.1.5, o sea es el `foto_mes` del registro desplazado k filas dentro del cliente
* NUNCA se modifican los valores del mes corriente
* si se modifica un lag, se recalcula su delta como  `valor actual - lag modificado`, para que delta y lag sean consistentes
* `kmes` (mes calendario) no se trata: su lag es calendario puro, no informacion afectada por la pandemia

In [37]:
# parametros del tratamiento de pandemia
PARAM$FE_pandemia$meses_pandemia <- c(202003, 202004, 202005, 202006, 202007)
PARAM$FE_pandemia$cols_excluidas <- c("kmes")

setorder( dataset, numero_de_cliente, foto_mes )

# variables cuyos lags pueden ser tratados: las lagueables numericas, excepto las excluidas
cols_tratadas <- cols_lagueables[ sapply(cols_lagueables, function(v) is.numeric(dataset[[v]])) ]
cols_tratadas <- setdiff( cols_tratadas, PARAM$FE_pandemia$cols_excluidas )
length(cols_tratadas)

# mes al que apunta cada lag (columnas auxiliares, se eliminan al final de esta seccion)
for (k in PARAM$FEhist$lags)
  dataset[, paste0("foto_mes_apuntado_lag", k) := shift(foto_mes, k, NA, "lag"), by = numero_de_cliente]

# foto de control del mes corriente, para verificar luego que no se modifico
control_mes_corriente <- dataset[, lapply(.SD, sum, na.rm=TRUE), .SDcols = cols_tratadas]

# cantidad de registros cuyo lag apunta a un mes de pandemia, por foto_mes
tb_apunta_pandemia <- dataset[, lapply(.SD, function(x) sum(x %in% PARAM$FE_pandemia$meses_pandemia)),
  by = foto_mes, .SDcols = paste0("foto_mes_apuntado_lag", PARAM$FEhist$lags)]
tb_apunta_pandemia[ rowSums(tb_apunta_pandemia[, -1]) > 0 ]

[1] 53

foto_mes,foto_mes_apuntado_lag1,foto_mes_apuntado_lag3,foto_mes_apuntado_lag6
<int>,<int>,<int>,<int>
202004,29654,0,0
202005,29741,0,0
202006,30009,29474,0
202007,30491,29526,0
202008,30900,29766,0
202009,1,30278,29145
202010,0,30739,29241
202011,2,2,29496
202012,0,1,29986


In [38]:
# Reemplaza el lag de orden k de las variables vcols en los registros cuyo lag apunta a un mes de pandemia
#   tb_ref tiene una fila por numero_de_cliente con el valor de reemplazo de cada variable
#   el delta se recalcula como  valor actual - lag reemplazado
#   devuelve la cantidad de registros modificados
ReemplazarLagPandemia <- function(k, tb_ref, vcols) {

  col_mes <- paste0("foto_mes_apuntado_lag", k)
  filas <- which( dataset[[col_mes]] %in% PARAM$FE_pandemia$meses_pandemia )
  idx <- match( dataset$numero_de_cliente[filas], tb_ref$numero_de_cliente )

  for (v in vcols) {
    vlag <- paste0(v, "_lag", k)
    vdelta <- paste0(v, "_delta", k)

    nuevo_lag <- tb_ref[[v]][idx]
    # si el reemplazo no es entero y la columna si, la paso a double para no truncar
    if( is.integer(dataset[[vlag]]) && !is.integer(nuevo_lag) )
      set( dataset, j= vlag, value= as.numeric(dataset[[vlag]]) )
    set( dataset, i= filas, j= vlag, value= nuevo_lag )

    nuevo_delta <- dataset[[v]][filas] - dataset[[vlag]][filas]
    if( is.integer(dataset[[vdelta]]) && !is.integer(nuevo_delta) )
      set( dataset, j= vdelta, value= as.numeric(dataset[[vdelta]]) )
    set( dataset, i= filas, j= vdelta, value= nuevo_delta )
  }

  return( length(filas) )
}

In [39]:
# 5B3  Valores prepandemia del mismo cliente
#   lag1 -> 202002    lag3 -> 201912    lag6 -> 201909
PARAM$FE_pandemia$mes_referencia <- list( lag1= 202002, lag3= 201912, lag6= 201909 )

for (k in PARAM$FEhist$lags)
{
  mes_ref <- PARAM$FE_pandemia$mes_referencia[[ paste0("lag", k) ]]

  # valor de cada variable para cada cliente en el mes de referencia
  #  si el cliente no existe en ese mes, el lag queda NA
  tb_ref <- dataset[ foto_mes == mes_ref, c("numero_de_cliente", cols_tratadas), with= FALSE ]

  n <- ReemplazarLagPandemia( k, tb_ref, cols_tratadas )
  cat( "lag", k, ": registros con lag reemplazado por el valor de", mes_ref, "=", n, "\n" )
}
rm( tb_ref )

lag 1 : registros con lag reemplazado por el valor de 202002 = 150803 
lag 3 : registros con lag reemplazado por el valor de 201912 = 149808 
lag 6 : registros con lag reemplazado por el valor de 201909 = 148297 


Verificacion de las features del experimento

In [40]:
# verificacion 5B3: por un camino independiente (join) se compara el lag con el valor del mes de referencia
v <- cols_tratadas[1]
for (k in PARAM$FEhist$lags)
{
  mes_ref <- PARAM$FE_pandemia$mes_referencia[[ paste0("lag", k) ]]
  col_mes <- paste0("foto_mes_apuntado_lag", k)
  tb_chk <- dataset[ get(col_mes) %in% PARAM$FE_pandemia$meses_pandemia,
                     list(numero_de_cliente, lag= get(paste0(v, "_lag", k))) ]
  tb_r <- dataset[ foto_mes == mes_ref, list(numero_de_cliente, r= get(v)) ]
  tb_chk <- tb_r[ tb_chk, on= "numero_de_cliente" ]
  stopifnot( isTRUE(all.equal( as.numeric(tb_chk$lag), as.numeric(tb_chk$r) )) )
}
cat("OK lags reemplazados por el valor prepandemia (variable", v, ")\n")

OK lags reemplazados por el valor prepandemia (variable internet )


In [41]:
# verificaciones comunes

# 1. el mes corriente no fue modificado
stopifnot( isTRUE(all.equal(
  control_mes_corriente,
  dataset[, lapply(.SD, sum, na.rm=TRUE), .SDcols = cols_tratadas] )) )
cat("OK mes corriente sin modificar\n")

# 2. consistencia delta = valor actual - lag, para todas las variables y ordenes de lag
max_error_delta <- 0
for (v in cols_tratadas)
  for (k in PARAM$FEhist$lags)
    max_error_delta <- max( max_error_delta, suppressWarnings(
      max( abs( dataset[[v]] - dataset[[paste0(v, "_lag", k)]] - dataset[[paste0(v, "_delta", k)]] ),
           na.rm= TRUE ) ) )
stopifnot( max_error_delta < 1e-6 )
cat("OK deltas consistentes con los lags, error maximo =", max_error_delta, "\n")

# elimino las columnas auxiliares, NO son features
dataset[, paste0("foto_mes_apuntado_lag", PARAM$FEhist$lags) := NULL ]
rm( control_mes_corriente )
gc(full= TRUE, verbose= FALSE)

OK mes corriente sin modificar
OK deltas consistentes con los lags, error maximo = 0 


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2162067,115.5,4088424,218.4,4088424,218.4
Vcells,300454667,2292.3,475486271,3627.7,396115803,3022.2


Campos en los que se entrena el modelo

In [42]:
# los campos en los que se entrena
#  se excluyen tambien las columnas de la particion, que ahora existen antes de este paso
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar","fold_train","fold_final_train"))
)

stopifnot( !any( grepl("^foto_mes_apuntado_lag", campos_buenos) ) )
stopifnot( !any( grepl("^pandemia_lag", campos_buenos) ) )
length(campos_buenos)

[1] 380

####  9.3.2.3 Hyperparameter Tuning

Construccion de los datasets de LightGBM
<br>usa la particion definida en 9.3.2.1 y las features `campos_buenos` armadas en 9.3.2.2

In [43]:
if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

Loading required package: lightgbm



In [44]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 32787

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * learning_rate = 0.03  para que aprenda lento
  * seed : se reemplaza por cada una de las 10 semillas del experimento

* Parametros que se optimizan en el Grid Search  (3 x 5 x 2 = 30 combinaciones)
  * num_leaves  { 32, 64, 128 }
  * min_data_in_leaf  { 512, 1024, 2048, 4096, 8192 }

Grilla de la segunda corrida (Second_Run). En la primera corrida (num_leaves 32 a 512, min_data_in_leaf 64 a 2048)
min_data_in_leaf quedo en el extremo superior (2048) en 25 de 40 semillas y la AUC media seguia subiendo,
mientras que num_leaves >= 256 y min_data_in_leaf <= 256 practicamente nunca fueron optimos.
<br>Se mantiene 512 para que 1024 quede en el interior de la grilla.
  * feature_fraction  { 0.5, 0.8 }

El Grid Search se repite para cada una de las 10 semillas y se eligen los mejores hiperparametros de CADA semilla con el criterio de validacion original (AUC en 202105).
<br>La AUC solo se usa para elegir hiperparametros; los experimentos se comparan por ganancia de produccion.

In [45]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 4096,  # valor grande, lo limita early_stopping_rounds (Second_Run: con min_data_in_leaf alto pueden hacer falta mas iteraciones)
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)

In [46]:
# avance del Grid Search, lo actualiza el loop de semillas de 9.3.2.3
progreso_ht <- new.env()
progreso_ht$texto <- ""
progreso_ht$corrida <- 0
progreso_ht$total <- NA
progreso_ht$corridas_hechas <- 0
progreso_ht$corridas_totales <- NA
progreso_ht$inicio <- Sys.time()

# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  # avance: corrida dentro de la semilla y tiempo estimado restante del experimento
  progreso_ht$corrida <- progreso_ht$corrida + 1
  progreso_ht$corridas_hechas <- progreso_ht$corridas_hechas + 1
  min_transcurridos <- as.numeric(difftime(Sys.time(), progreso_ht$inicio, units= "mins"))
  min_restantes <- min_transcurridos / progreso_ht$corridas_hechas *
    (progreso_ht$corridas_totales - progreso_ht$corridas_hechas)

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    progreso_ht$texto, "  corrida ", progreso_ht$corrida, "/", progreso_ht$total,
    "  (", progreso_ht$corridas_hechas, "/", progreso_ht$corridas_totales, " total, faltan ~",
    round(min_restantes), " min)  ",
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  # hago espacio en la memoria
  niter <- modelo_train$best_iter
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [47]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
PARAM$HT$num_leaves <- c(32, 64, 128)
PARAM$HT$min_data_in_leaf <- c(512, 1024, 2048, 4096, 8192)
PARAM$HT$feature_fraction <- c(0.5, 0.8)

tb_grilla <- CJ(
  num_leaves= PARAM$HT$num_leaves,
  min_data_in_leaf= PARAM$HT$min_data_in_leaf,
  feature_fraction= PARAM$HT$feature_fraction
)

stopifnot( nrow(tb_grilla) == 30 )

Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda corre 30 combinaciones x 10 semillas = 300 entrenamientos, aproximadamente 6 veces lo que demoraba la notebook original
<br> Cada semilla graba su tabla apenas termina (`tb_grid_search_s<semilla>_5B3_Second_Run.txt`). Si la corrida se interrumpe, al volver a ejecutar la celda se leen las semillas ya terminadas y no se recalculan. Para forzar el recalculo, borrar esos archivos.

In [ ]:
tb_grid_semillas <- list()

# semillas que faltan calcular (las ya grabadas se leen del checkpoint)
semillas_pendientes <- PARAM$semillas[ !file.exists(
  paste0("tb_grid_search_s", PARAM$semillas, PARAM$sufijo, ".txt") ) ]
progreso_ht$corridas_totales <- nrow(tb_grilla) * length(semillas_pendientes)
progreso_ht$corridas_hechas <- 0
progreso_ht$inicio <- Sys.time()

cat( "Grid Search", PARAM$experimento, ":", length(semillas_pendientes), "semillas a calcular,",
     progreso_ht$corridas_totales, "corridas en total\n" )

for (semilla_actual in PARAM$semillas)
{
  archivo_grid_semilla <- paste0("tb_grid_search_s", semilla_actual, PARAM$sufijo, ".txt")
  nro_semilla <- match(semilla_actual, PARAM$semillas)

  if( file.exists(archivo_grid_semilla) ) {
    cat( "semilla ", nro_semilla, "/", length(PARAM$semillas), ": ", semilla_actual,
         "  ya calculada, se lee ", archivo_grid_semilla, "\n", sep="" )
    tb_nueva <- fread( archivo_grid_semilla )
    stopifnot( nrow(tb_nueva) == nrow(tb_grilla), !anyNA(tb_nueva$AUC) )

  } else {

    cat( "semilla ", nro_semilla, "/", length(PARAM$semillas), ": ", semilla_actual,
         "  inicio Grid Search (", nrow(tb_grilla), " corridas)\n", sep="" )

    # avance que muestra Estimar_AUC_lightgbm en cada corrida
    progreso_ht$texto <- paste0("semilla ", nro_semilla, "/", length(PARAM$semillas), ": ", semilla_actual)
    progreso_ht$corrida <- 0
    progreso_ht$total <- nrow(tb_grilla)

    # la semilla de LightGBM
    PARAM$lgbm$param_fijos$seed <- semilla_actual

    # si hubiera undersampling de los CONTINUA, el azar depende de la semilla
    #  con training_pct = 1.0 el training es el mismo para todas las semillas
    if( PARAM$trainingstrategy$training_pct < 1.0 ) {
      set.seed(semilla_actual, kind = "L'Ecuyer-CMRG")
      dataset[, azar:=runif(nrow(dataset))]
      dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
          (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
           azar < PARAM$trainingstrategy$training_pct ) ]
      dtrain <- lgb.Dataset(
        data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
        label= dataset[fold_train == TRUE, clase01],
        free_raw_data= TRUE
      )
    }

    tb_nueva <- copy( tb_grilla )

    # registro a registro calculo la AUC
    tb_nueva[,  c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
      by=1:nrow(tb_nueva) ]

    tb_nueva[, semilla := semilla_actual ]

    # checkpoint de la semilla
    fwrite( tb_nueva,
      file= archivo_grid_semilla,
      sep="\t"
    )
  }

  tb_grid_semillas[[ as.character(semilla_actual) ]] <- tb_nueva
}

# vuelvo la semilla de param_fijos a la primigenia
PARAM$lgbm$param_fijos$seed <- PARAM$semilla_primigenia

Grid Search 5B3 : 10 semillas a calcular, 300 corridas en total
semilla 1/10: 701669  inicio Grid Search (30 corridas)


Tue Nov 03 01:27:48 2026  semilla 1/10: 701669  corrida 1/30  (1/300 total, faltan ~549 min)  32, 512, 0.5 niter 426 AUC 0.931009841943594

Tue Nov 03 01:28:58 2026  semilla 1/10: 701669  corrida 2/30  (2/300 total, faltan ~446 min)  32, 512, 0.8 niter 704 AUC 0.92996312247624

Tue Nov 03 01:30:47 2026  semilla 1/10: 701669  corrida 3/30  (3/300 total, faltan ~476 min)  32, 1024, 0.5 niter 460 AUC 0.931042026243647

Tue Nov 03 01:31:52 2026  semilla 1/10: 701669  corrida 4/30  (4/300 total, faltan ~436 min)  32, 1024, 0.8 niter 565 AUC 0.931896446438488

Tue Nov 03 01:33:40 2026  semilla 1/10: 701669  corrida 5/30  (5/300 total, faltan ~454 min)  32, 2048, 0.5 niter 462 AUC 0.932500996637931

Tue Nov 03 01:34:31 2026  semilla 1/10: 701669  corrida 6/30  (6/300 total, faltan ~419 min)  32, 2048, 0.8 niter 358 AUC 0.930535181126942



la optimizacion de hiperparámetros de tipo  Grid Search ha corrido para las 10 semillas, grabo los resultados y extraigo los mejores hiperparametros de cada semilla

In [ ]:
# resultados completos del Grid Search, todas las semillas
tb_grid_search <- rbindlist( tb_grid_semillas, use.names= TRUE )
setcolorder( tb_grid_search, "semilla" )

stopifnot( nrow(tb_grid_search) == nrow(tb_grilla) * length(PARAM$semillas) )

tb_grid_search

# se graba sin append, para no duplicar filas si la notebook se re-ejecuta
fwrite( tb_grid_search,
  file= paste0("tb_grid_search", PARAM$sufijo, ".txt"),
  sep="\t"
)

In [ ]:
# para CADA semilla, ordeno DESCENDENTE por AUC, en la posicion 1 estan los mejores
setorder( tb_grid_search, semilla, -AUC )
tb_mejores <- tb_grid_search[, .SD[1], by= semilla ]
tb_mejores <- tb_mejores[ match(PARAM$semillas, semilla) ]

# advertencia si el optimo esta en un extremo de la grilla
tb_mejores[, extremo_grilla :=
  num_leaves %in% range(PARAM$HT$num_leaves) |
  min_data_in_leaf %in% range(PARAM$HT$min_data_in_leaf) ]

for( i in which(tb_mejores$extremo_grilla) )
  warning( "semilla ", tb_mejores$semilla[i],
           ": el optimo (num_leaves=", tb_mejores$num_leaves[i],
           ", min_data_in_leaf=", tb_mejores$min_data_in_leaf[i],
           ") esta en un extremo de la grilla, podria convenir ampliar la busqueda",
           call.= FALSE, immediate.= TRUE )

cat( "semillas con optimo en un extremo de la grilla:", sum(tb_mejores$extremo_grilla), "de", nrow(tb_mejores), "\n" )

# advertencia si alguna corrida no llego a cortar por early stopping (iteraciones en el borde)
n_iter_borde <- tb_grid_search[ num_iterations >= 0.9 * PARAM$lgbm$param_fijos$num_iterations, .N ]
if( n_iter_borde > 0 )
  warning( n_iter_borde, " corridas con num_iterations cerca del tope de ", PARAM$lgbm$param_fijos$num_iterations,
           ", podria convenir subir num_iterations", call.= FALSE, immediate.= TRUE )
cat( "maximo num_iterations en el Grid Search:", max(tb_grid_search$num_iterations), "\n" )

tb_mejores

In [ ]:
# grabo los mejores hiperparametros de cada semilla, Produccion los lee de este archivo
fwrite( tb_mejores,
  file= paste0("tb_mejores_hiperparametros", PARAM$sufijo, ".txt"),
  sep="\t"
)

PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_mejores )

if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file= paste0("PARAM_HT", PARAM$sufijo, ".yml") )

### 9.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en el Grid Search

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)

# libero la memoria de los datasets del Hyperparameter Tuning
if( exists("dtrain") ) rm(dtrain)
if( exists("dvalidate") ) rm(dvalidate)
gc(full= TRUE, verbose= FALSE)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

##### Final Training Hyperparameters

In [ ]:
# recupero los mejores hiperparametros grabados por el Hyperparameter Tuning (9.3.2.3)
archivo_mejores <- paste0("tb_mejores_hiperparametros", PARAM$sufijo, ".txt")

if( !file.exists(archivo_mejores) )
  stop("No existe ", archivo_mejores, " , primero debe correr 9.3.2.3 Hyperparameter Tuning")

tb_mejores_hp <- fread( archivo_mejores )

stopifnot( nrow(tb_mejores_hp) == length(PARAM$semillas),
           setequal(tb_mejores_hp$semilla, PARAM$semillas) )

tb_mejores_hp <- tb_mejores_hp[ match(PARAM$semillas, semilla) ]
tb_mejores_hp

In [ ]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en el Grid Search
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con el Grid Search, uno por semilla
#  IMPORTANTE: se usa modifyList() y no c()
#  c() dejaba nombres duplicados (num_leaves, min_data_in_leaf, feature_fraction) y
#  LightGBM se queda con el PRIMER valor, o sea el fijo, ignorando el optimizado
param_final <- list()
for( i in seq_len(nrow(tb_mejores_hp)) )
{
  hp <- tb_mejores_hp[i]
  param_final[[ as.character(hp$semilla) ]] <- modifyList( fijos, list(
    seed= hp$semilla,
    num_leaves= hp$num_leaves,
    min_data_in_leaf= hp$min_data_in_leaf,
    feature_fraction= hp$feature_fraction,
    num_iterations= as.integer(hp$num_iterations)
  ))
}

# verificacion: sin nombres duplicados y con los valores optimizados
for( s in names(param_final) )
  stopifnot( !anyDuplicated(names(param_final[[s]])),
             param_final[[s]]$num_leaves == tb_mejores_hp[semilla == as.numeric(s), num_leaves],
             param_final[[s]]$num_iterations == tb_mejores_hp[semilla == as.numeric(s), num_iterations] )

param_final[[1]]

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria
<br>Un modelo por semilla, cada uno con sus mejores hiperparametros y su cantidad de iteraciones

In [ ]:
final_model <- list()

for( s in names(param_final) )
{
  cat( format(Sys.time(), "%a %b %d %X %Y  "), "semilla", s, "\n" )

  final_model[[s]] <- lgb.train(
    data= dfinal_train,
    param= param_final[[s]],
    verbose= -100
  )

  # verifico que se entreno con la cantidad de iteraciones optimizada
  stopifnot( final_model[[s]]$current_iter() == param_final[[s]]$num_iterations )
}

In [ ]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

for( s in names(final_model) )
  lgb.save(final_model[[s]], paste0("modelo_s", s, PARAM$sufijo, ".txt"))

In [ ]:
# ahora imprimo la importancia de variables

for( s in names(final_model) )
{
  tb_importancia <- as.data.table(lgb.importance(final_model[[s]]))
  archivo_importancia <- paste0("impo_s", s, PARAM$sufijo, ".txt")

  fwrite( tb_importancia,
    file= archivo_importancia,
    sep= "\t"
  )
}

#### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
PARAM$trainingstrategy$future <- c(202107)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [ ]:
# aplico final_model   a dfuture, una prediccion por semilla

mfuture <- data.matrix(dfuture[, campos_buenos, with= FALSE])

prediccion <- list()
for( s in names(final_model) )
  prediccion[[s]] <- predict(
    final_model[[s]],
    mfuture
  )

##### Tabla Prediccion

In [ ]:
tb_prediccion_semilla <- list()

for( s in names(prediccion) )
{
  tb_prediccion <- dfuture[, list(numero_de_cliente, clase_ternaria)]
  tb_prediccion[, prob := prediccion[[s]] ]

  # grabo las probabilidad del modelo
  #  me va a ser util para hacer Ensembles de modelos
  fwrite(tb_prediccion,
    file= paste0("prediccion_s", s, PARAM$sufijo, ".txt"),
    sep= "\t"
  )

  tb_prediccion_semilla[[s]] <- tb_prediccion
}

#### Curva de Ganancia

Genero las salidas

In [ ]:
for( s in names(tb_prediccion_semilla) )
{
  tb_prediccion <- tb_prediccion_semilla[[s]]

  # asigno clase ternaria
  tb_prediccion[, clase_ternaria := dfuture$clase_ternaria ]

  # asigno ganancias
  tb_prediccion[, ganancia := -0.025 ]
  tb_prediccion[clase_ternaria=="BAJA+2", ganancia := 0.975 ]

  # reordeno, y acumulada
  setorder( tb_prediccion, -prob )
  tb_prediccion[, gan_acum := cumsum(ganancia)]

  # media movil de ancho 400
  tb_prediccion[,
    gan_suavizada := frollmean(
      x= gan_acum,
      n= 400,
      align= "center",
      na.rm= TRUE,
      hasNA= TRUE
    )
  ]

  tb_prediccion_semilla[[s]] <- tb_prediccion
}

In [ ]:
for( s in names(tb_prediccion_semilla) )
  print( c( semilla= as.numeric(s),
            no_NA= tb_prediccion_semilla[[s]][, sum(!is.na(gan_suavizada) )],
            N= tb_prediccion_semilla[[s]][, .N] ) )

In [ ]:
# Este es la ganancia a reportar, una por semilla
resultado_semilla <- list()
for( s in names(tb_prediccion_semilla) )
{
  tb_prediccion <- tb_prediccion_semilla[[s]]
  resultado <- list()
  resultado$ganancia_suavizada_max <- max( tb_prediccion$gan_suavizada, na.rm=TRUE )
  resultado$envios <- which.max( tb_prediccion$gan_suavizada)
  resultado_semilla[[s]] <- resultado
}
options(digits= 8)

tb_resultados <- data.table(
  experimento= PARAM$experimento,
  semilla= as.numeric(names(resultado_semilla)),
  ganancia_suavizada_max= sapply(resultado_semilla, function(r) r$ganancia_suavizada_max),
  envios= sapply(resultado_semilla, function(r) r$envios)
)

stopifnot( nrow(tb_resultados) == 10, identical(tb_resultados$semilla, PARAM$semillas) )

tb_resultados
summary( tb_resultados$ganancia_suavizada_max )

In [ ]:
# grabo las ganancias
for( s in names(tb_prediccion_semilla) )
  fwrite( tb_prediccion_semilla[[s]],
    file= paste0("ganancias_s", s, PARAM$sufijo, ".txt"),
    sep= "\t"
  )

fwrite( tb_resultados,
  file= paste0("ganancias_produccion", PARAM$sufijo, ".txt"),
  sep= "\t"
)

In [ ]:
# genero el grafico, uno por semilla
#  ylim se adapta a los datos: con ganancias de 0.975 / -0.025 el ylim original c(0, 15000000) dejaba la curva aplastada

for( s in names(tb_prediccion_semilla) )
{
  tb_prediccion <- tb_prediccion_semilla[[s]]
  resultado <- resultado_semilla[[s]]

  tb_prediccion[, envios:= .I]

  pdf(paste0("curva_de_ganancia_s", s, PARAM$sufijo, ".pdf"))

  plot(
    x= tb_prediccion$envios,
    y= tb_prediccion$gan_acum,
    type= "l",
    col= "gray",
    xlim= c(0, 10000),
    ylim= c(0, max(tb_prediccion$gan_acum, na.rm=TRUE) * 1.05),
    main= paste0(PARAM$experimento, " semilla ", s,
                 "  Mejor gan prom= ", round(resultado$ganancia_suavizada_max, 2), " envios= ", resultado$envios),
    xlab= "Envios",
    ylab= "Ganancia",
    panel.first= grid()
  )

  dev.off()
}

In [ ]:
# grafico conjunto: las 10 curvas de ganancia acumulada del experimento

pdf(paste0("curva_de_ganancia", PARAM$sufijo, ".pdf"))

ymax <- max( sapply(tb_prediccion_semilla, function(t) max(t$gan_acum, na.rm=TRUE)) ) * 1.05
colores <- rainbow( length(tb_prediccion_semilla) )

plot( NULL,
  xlim= c(0, 10000),
  ylim= c(0, ymax),
  main= paste0(PARAM$experimento, "  ganancia media= ", round(mean(tb_resultados$ganancia_suavizada_max), 2)),
  xlab= "Envios",
  ylab= "Ganancia",
  panel.first= grid()
)

for( i in seq_along(tb_prediccion_semilla) )
  lines( x= tb_prediccion_semilla[[i]]$envios, y= tb_prediccion_semilla[[i]]$gan_acum, col= colores[i] )

legend( "bottomright", legend= names(tb_prediccion_semilla), col= colores, lty= 1, cex= 0.6 )

dev.off()

In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

# vector de 10 ganancias de produccion, en el orden de PARAM$semillas
PARAM$resultado <- list(
  semillas= tb_resultados$semilla,
  ganancias= tb_resultados$ganancia_suavizada_max,
  envios= tb_resultados$envios,
  ganancia_media= mean(tb_resultados$ganancia_suavizada_max)
)

write_yaml( PARAM, file= paste0("PARAM", PARAM$sufijo, ".yml") )

# archivo chico solo con las ganancias, para la notebook de comparacion (Wilcoxon)
write_yaml( list(
    experimento= PARAM$experimento,
    semillas= tb_resultados$semilla,
    ganancias= tb_resultados$ganancia_suavizada_max ),
  file= paste0("ganancias_produccion", PARAM$sufijo, ".yml") )

# verificacion de lo grabado
chk <- read_yaml( paste0("ganancias_produccion", PARAM$sufijo, ".yml") )
stopifnot( length(chk$ganancias) == 10, identical(as.numeric(chk$semillas), PARAM$semillas) )
chk$ganancias

In [ ]:
# archivos generados por este experimento
list.files( pattern= PARAM$sufijo )

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")